<a href="https://colab.research.google.com/github/Adhira-Deogade/pytorch-learnings/blob/main/notebooks/sd/smalldiffusion_implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!git clone https://github.com/yuanchenyang/smalldiffusion.git

Cloning into 'smalldiffusion'...
remote: Enumerating objects: 405, done.
remote: Counting objects: 100% (106/106), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 405 (delta 80), reused 85 (delta 76), pack-reused 299 (from 1)
Receiving objects: 100% (405/405), 1.78 MiB | 4.84 MiB/s, done.
Resolving deltas: 100% (199/199), done.


In [4]:
!pip install torch_ema

In [5]:
# import sys
# import os

# # Change directory to the cloned smalldiffusion repository
# %cd /content/smalldiffusion

# # Add the current directory to the system path to allow importing smalldiffusion
# sys.path.append(os.getcwd())

In [6]:
import torch
from accelerate import Accelerator
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
from torchvision.utils import make_grid, save_image
from torch_ema import ExponentialMovingAverage as EMA
from tqdm import tqdm
from smalldiffusion.src.smalldiffusion import (
    Unet, Scaled, ScheduleLogLinear, ScheduleSigmoid, samples, training_loop,
    MappedDataset, img_train_transform, img_normalize
)


In [7]:
cifar10_dataset = MappedDataset(
      CIFAR10(
          'datasets',
          train=True,
          download=True,
          transform=img_train_transform),
      lambda x:x[0]
  )

100%|██████████| 170M/170M [00:05<00:00, 28.9MB/s]


In [8]:
print(type(cifar10_dataset))
print(cifar10_dataset)
# for i in range(10):
#   print(cifar10_dataset[i])
print(len(cifar10_dataset))
print(cifar10_dataset[0].shape)

<class 'smalldiffusion.src.smalldiffusion.data.MappedDataset'>
50000
torch.Size([3, 32, 32])


In [9]:
def main(
    train_batch_size=256,
    epochs=1000,
    sample_batch_size=64):
  # Setup
  gpu_accelerator = Accelerator()
  cifar10_dataset = MappedDataset(
      CIFAR10(
          'datasets',
          train=True,
          download=True,
          transform=img_train_transform),
      lambda x:x[0]
  )
  cifar10_dataloader = DataLoader(
      cifar10_dataset,
      batch_size=train_batch_size,
      shuffle=True
  )
  noise_schedule = ScheduleSigmoid(N=1000)
  model = Scaled(Unet)(
      32, 3, 3, ch=128, ch_mult=(1,2,2,2),
      attn_resolutions=(16, )
  )
  # Train
  ema = EMA(model.parameters(), decay=0.9999)
  ema.to(gpu_accelerator.device)
  for ns in training_loop(
      cifar10_dataloader,
      model,
      noise_schedule,
      epochs=epochs,
      lr=2e-4,
      accelerator=gpu_accelerator,
  ):
    ns.pbar.set_description(f"Loss={ns.loss.item():.5}")
    ema.update()
  # Sampling
  sample_schedule = ScheduleLogLinear(sigma_min=0.02, sigma_max=35, N=1000)
  with ema.average_parameters():
    *xt, x0 = samples(
        model,
        sample_schedule.sample_sigmas(10),
        gam=2.1,
        batch_size=sample_batch_size,
        accelerator=gpu_accelerator,
    )
    save_image(img_normalize(make_grid(x0)), 'samples.png')
    torch.save(model.state_dict(), 'checkpoint.pt')



In [ ]:
if __name__=='__main__':
  main()

Loss=0.13745:   0%|          | 0/1000 [01:20<?, ?it/s]